# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/peddikotlachiranjeevi/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Selected lane: Lane 2 ? Refresh / Content Opportunity Scoring**

I selected this lane because content teams need a practical way to identify which existing pages should be reviewed first. The starter dataset contains search performance, traffic, engagement, content age, and freshness signals. These signals can help identify pages that may deserve attention. The intended output is a prioritized review queue for content strategists and editors. I will begin with exploratory analysis and a rule-based baseline. I will only use machine learning if it demonstrates value beyond a simple rule.

In [13]:
# Load the starter dataset and verify its structure.
import csv
from pathlib import Path

possible_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]

DATA_PATH = next((p for p in possible_paths if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Starter CSV not found. Run this notebook from the repository "
        "root or update DATA_PATH."
    )

with DATA_PATH.open("r", encoding="utf-8-sig", newline="") as file:
    reader = csv.DictReader(file)
    rows = list(reader)
    columns = reader.fieldnames

required_columns = {
    "content_id", "client_id", "impressions_90d",
    "sessions_90d", "trend_direction", "trend_pct",
    "impressions_last_30d", "impressions_prev_30d",
    "content_age_days", "avg_position"
}

missing_columns = required_columns.difference(columns or [])

if missing_columns:
    raise ValueError(f"Missing expected columns: {sorted(missing_columns)}")

print("Dataset loaded successfully")
print("File:", DATA_PATH)
print("Number of records:", len(rows))
print("Number of columns:", len(columns))
print("Number of represented clients:", len({r["client_id"] for r in rows}))
print("One row represents: one pseudonymized content item")

Dataset loaded successfully
File: ..\..\data\raw\content_refresh_anonymized.csv
Number of records: 30000
Number of columns: 44
Number of represented clients: 32
One row represents: one pseudonymized content item


## 2. The question: decision, action, cost of a wrong call

**Research question:** How can we prioritize existing content pages for human review using available search performance, traffic, engagement, and freshness signals?

- **Decision:** Which content pages should a team review first?
- **Who acts:** Content strategists, SEO specialists, and editors.
- **Action:** Review prioritized pages and decide whether to refresh, expand, protect, monitor, or leave them unchanged.
- **Cost of a false positive:** Editorial time is wasted reviewing a page that may not need changes.
- **Cost of a false negative:** A page that deserves attention may be overlooked.
- **Task type:** Ranking / scoring.
- **Proposed evaluation metric:** Precision@K, once an independently observed future outcome is available.

The current starter snapshot does not provide a verified future outcome for this ranking task. Therefore, Precision@K cannot yet establish future performance. A rule-based ranking will be the initial baseline, and the target and evaluation window must be defined before training a predictive model.

In [14]:
# Record the decision frame and the evaluation requirements.
research_frame = {
    "lane": "Lane 2 - Refresh / Content Opportunity Scoring",
    "decision": "Which content pages should a human reviewer prioritize?",
    "users": "Content strategists, SEO specialists, and editors",
    "output": "Prioritized content review queue",
    "task_type": "Ranking / scoring",
    "proposed_metric": "Precision@K",
    "initial_baseline": "Transparent rule-based ranking",
    "current_target_status": "Future outcome not established in the starter snapshot",
}

for key, value in research_frame.items():
    print(f"{key.replace('_', ' ').title()}: {value}")

print("\nFraming check:")
print("- The decision and intended user are specified.")
print("- The starter snapshot is not treated as a future outcome.")
print("- A baseline must be compared before claiming ML adds value.")

Lane: Lane 2 - Refresh / Content Opportunity Scoring
Decision: Which content pages should a human reviewer prioritize?
Users: Content strategists, SEO specialists, and editors
Output: Prioritized content review queue
Task Type: Ranking / scoring
Proposed Metric: Precision@K
Initial Baseline: Transparent rule-based ranking
Current Target Status: Future outcome not established in the starter snapshot

Framing check:
- The decision and intended user are specified.
- The starter snapshot is not treated as a future outcome.
- A baseline must be compared before claiming ML adds value.


## 3. Quick look at the data (2-3 real numbers)

The starter dataset contains **30,000 anonymized content records** and **44 columns**, with one row per content item.

My initial analysis found:

- **16,726 records** have at least 500 impressions in the trailing 90-day period.
- **16,262 records (54.21%)** have `trend_direction = down`.
- **7,114 records** have at least 30 sessions in the trailing 90-day period.

These figures show that the dataset contains a substantial number of pages with measurable search exposure and traffic. The downward trend category is a snapshot-derived indicator, not an independently observed future decline.

In [15]:
# Calculate reproducible dataset-level numbers.
def number(row, column):
    value = row.get(column, "")
    return float(value) if value not in ("", None) else 0.0

total = len(rows)
declining = sum(row["trend_direction"] == "down" for row in rows)
high_impressions = sum(number(row, "impressions_90d") >= 500 for row in rows)
active_sessions = sum(number(row, "sessions_90d") >= 30 for row in rows)

if total == 0:
    raise ValueError("The starter dataset contains no records.")

print("STARTER DATASET SUMMARY")
print("-----------------------")
print(f"Content records: {total:,}")
print(f"Records with 500+ 90-day impressions: {high_impressions:,}")
print(f"Records with 30+ 90-day sessions: {active_sessions:,}")
print(
    f"Records marked down by the snapshot trend indicator: "
    f"{declining:,} ({declining / total * 100:.2f}%)"
)

print("\nTrend category distribution:")
from collections import Counter
for category, count in sorted(Counter(r["trend_direction"] for r in rows).items()):
    print(f"{category}: {count:,}")

print("\nNote: trend_direction is a derived snapshot indicator, not a future label.")

STARTER DATASET SUMMARY
-----------------------
Content records: 30,000
Records with 500+ 90-day impressions: 16,726
Records with 30+ 90-day sessions: 7,114
Records marked down by the snapshot trend indicator: 16,262 (54.21%)

Trend category distribution:
down: 16,262
flat: 1,152
new: 2,236
stable: 5,962
up: 4,388

Note: trend_direction is a derived snapshot indicator, not a future label.


## 4. Careful words: what I can and can't claim

**What I can claim**

- The starter dataset contains measurable search performance, traffic, engagement, and content freshness signals.
- Some records show lower recent performance compared with their previous 30-day period.
- These signals can support exploratory analysis and an initial human-review prioritization approach.

**What I cannot claim**

- I cannot claim that this snapshot predicts future traffic decline or recovery.
- I cannot claim that refreshing a page causes better rankings or traffic.
- I cannot claim that the results explain Google's ranking algorithm.
- I cannot use `trend_direction` or `trend_pct` as model input features because they are derived from the trend label.

The project will begin with exploratory analysis and a rule-based baseline. Any future predictive model must use a clearly defined, independently observed outcome from a later time window and leakage-safe validation. Results will be described as decision support, not causal proof.

In [16]:
# Run checks for important data and leakage precautions.
assert len(rows) == 30000, f"Expected 30,000 records, found {len(rows)}"
assert len(columns) == 44, f"Expected 44 columns, found {len(columns)}"

# IDs are for grouping and joining, never model input features.
identifier_columns = {"content_id", "client_id"}

# These fields are derived from the trend label and must not be features.
leakage_columns = {"trend_direction", "trend_pct"}

assert identifier_columns.issubset(columns)
assert leakage_columns.issubset(columns)

zero_position_count = sum(number(row, "avg_position") == 0 for row in rows)

print("DATA AND LEAKAGE CHECKS")
print("-----------------------")
print("Record count check: PASS")
print("Column count check: PASS")
print("Identifier columns identified for exclusion: PASS")
print("Derived trend columns identified for exclusion: PASS")
print(f"Rows where avg_position is 0 (treated as no data): {zero_position_count:,}")

print("\nRules for later modeling:")
print("- Do not use content_id or client_id as model features.")
print("- Do not use trend_direction or trend_pct as model features.")
print("- Treat avg_position = 0 as missing/no data, not rank zero.")
print("- Do not blindly replace all missing values with zero.")
print("- Use client_id for grouped splitting where appropriate.")
print("- Define a genuinely future outcome before predictive modeling.")

print("\nAll initial checks completed.")

DATA AND LEAKAGE CHECKS
-----------------------
Record count check: PASS
Column count check: PASS
Identifier columns identified for exclusion: PASS
Derived trend columns identified for exclusion: PASS
Rows where avg_position is 0 (treated as no data): 1,205

Rules for later modeling:
- Do not use content_id or client_id as model features.
- Do not use trend_direction or trend_pct as model features.
- Treat avg_position = 0 as missing/no data, not rank zero.
- Do not blindly replace all missing values with zero.
- Use client_id for grouped splitting where appropriate.
- Define a genuinely future outcome before predictive modeling.

All initial checks completed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.